# КИМ 1.1. Нейрон и перцептрон — эталонное решение

> **Это эталонное решение** для преподавателя. Студентам выдаётся ноутбук-задание
> [`kim-01-neuron-perceptron.ipynb`](./kim-01-neuron-perceptron.ipynb) без заполненных ячеек.
>
> Код ниже — один из возможных вариантов решения; не единственный и не обязательно
> оптимальный. Приводится для сверки и подготовки к защите.

---
## Часть А. Нейрон на чистом NumPy

### 0. Импорт библиотек

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.random.seed(42)

### 1. Прямой проход одного нейрона

In [ ]:
def neuron_forward(x, w, b, activation):
    z = np.dot(w, x) + b
    return activation(z)

### 2. Функции активации

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def tanh(z):
    return np.tanh(z)

def relu(z):
    return np.maximum(0, z)

### 3. Сравнение функций активации

In [ ]:
x_test = np.array([1.0, 2.0, -1.0, 0.5])
w_test = np.array([0.5, -0.5, 1.0, 0.25])
b_test = 0.1

print("sigmoid:", neuron_forward(x_test, w_test, b_test, sigmoid))
print("tanh:   ", neuron_forward(x_test, w_test, b_test, tanh))
print("relu:   ", neuron_forward(x_test, w_test, b_test, relu))

# Графики активаций
z = np.linspace(-5, 5, 200)
plt.figure(figsize=(8, 5))
plt.plot(z, sigmoid(z), label='sigmoid')
plt.plot(z, tanh(z), label='tanh')
plt.plot(z, relu(z), label='relu')
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
plt.xlabel('z'); plt.ylabel('f(z)'); plt.title('Функции активации')
plt.legend(); plt.grid(True); plt.show()

**Ответ:** `sigmoid` насыщается (производная → 0) при $|z| \gtrsim 4$, где
$\sigma(z) \to 0$ или $1$. Это проблематично, потому что при насыщении градиент
обнуляется и обучение через backprop замедляется или останавливается (проблема
затухающего градиента). `relu` не насыщается в положительной области
($f'(z) = 1$ при $z > 0$), поэтому градиент течёт свободно — основной повод
предпочесть `relu` в скрытых слоях глубоких сетей.

---
## Часть Б. Перцептрон на Keras (Fashion-MNIST)

### 4. Загрузка Fashion-MNIST и подготовка данных

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, utils

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# 28x28 -> 784, нормировка в [0, 1], one-hot метки
x_train = x_train.reshape(60000, 784).astype('float32') / 255.0
x_test  = x_test.reshape(10000, 784).astype('float32') / 255.0
y_train_oh = utils.to_categorical(y_train, 10)
y_test_oh  = utils.to_categorical(y_test, 10)

class_names = ['Футболка', 'Брюки', 'Свитер', 'Платье', 'Пальто',
               'Сандали', 'Рубашка', 'Кроссовки', 'Сумка', 'Ботильоны']
print(x_train.shape, x_test.shape, y_train_oh.shape)

### 5. Построение перцептрона

In [ ]:
model = keras.Sequential([
    layers.Dense(800, input_dim=784, activation='relu', name='hidden'),
    layers.Dense(10, activation='softmax', name='output'),
])
model.summary()

### 6. Компиляция

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

### 7. Обучение

In [ ]:
history = model.fit(x_train, y_train_oh,
                    batch_size=200, epochs=10,
                    validation_split=0.2, verbose=2)

### 8. Оценка точности на тесте

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_oh, verbose=0)
print(f'Тестовая точность: {test_acc:.4f}')
# Ожидается ~0.88

### 9. Визуализация предсказаний

In [ ]:
predictions = model.predict(x_test, verbose=0)

n = 8
plt.figure(figsize=(14, 3))
for i in range(n):
    plt.subplot(1, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    pred = np.argmax(predictions[i])
    true = y_test[i]
    color = 'green' if pred == true else 'red'
    plt.title(f'{class_names[pred][:6]}\n({class_names[true][:6]})', color=color, fontsize=9)
    plt.axis('off')
plt.suptitle('Зелёный — верно, красный — ошибка'); plt.tight_layout(); plt.show()

---
**Параметры сети `Dense(800) → Dense(10)`:** 784·800 + 800 = 628 000 на скрытом
слое; 800·10 + 10 = 8 010 на выходном — всего ~636 000 обучаемых параметров.